# U3-04 · Biclustering, variedades y cesta de la compra

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 3, apartados 3.4 a 3.6*

Ejecuta cada celda con **Mayús + Intro**. En el navegador t-SNE e Isomap tardan alrededor de un minuto.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su preparación se explica en `LEEME.md`:
> Cardoso, M. (2013). *Wholesale customers*. https://doi.org/10.24432/C5030X ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Biclustering de clientes y productos (3.4)

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import make_biclusters
from sklearn.cluster import SpectralCoclustering
from sklearn.metrics import consensus_score

# 1) datos sintéticos con 4 bloques ocultos y filas barajadas
D, filas, cols = make_biclusters((300, 40), 4, noise=5,
                                 shuffle=True, random_state=0)
sc = SpectralCoclustering(4, random_state=0).fit(D)
print("acierto sintético:", round(consensus_score(
    sc.biclusters_, (filas, cols)), 3))

# 2) clientes x productos: en cuántas facturas compró cada uno
t = pd.read_csv("cestas.csv")
nombres = pd.read_csv("productos.csv", index_col="producto")
A = pd.crosstab(t.cliente, t.producto)
A = A[A.sum(axis=1) >= 20]
print("matriz", A.shape)
sc = SpectralCoclustering(6, random_state=0).fit(np.log1p(A))
for b in range(6):
    f, p = sc.get_indices(b)
    dentro = A.iloc[f, p].values.mean()
    fuera = A.iloc[f].drop(columns=A.columns[p]).values.mean()
    print(b, len(f), "clientes", len(p), "productos",
          round(dentro, 2), round(fuera, 2))
    print("   ", "; ".join(nombres.descripcion[A.columns[p][:3]]))

import matplotlib.pyplot as plt
orden = A.iloc[np.argsort(sc.row_labels_),
               np.argsort(sc.column_labels_)]
plt.figure(figsize=(4.4, 2.4))
plt.imshow(np.log1p(orden.values), aspect="auto",
           interpolation="nearest")
plt.xlabel("100 productos"); plt.ylabel("1697 clientes")
plt.show()  # plt.savefig("bicluster.pdf")

**Prueba.** Cambia el número de bloques de 6 a 4 y a 8. ¿Siguen teniendo un tema reconocible?

## 2. Aprendizaje de variedades (3.5)

La primera celda prepara los datos de los mayoristas como en el apartado 3.2.

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
m = pd.read_csv("mayoristas.csv")
gasto = m.drop(columns=["canal", "region"])
X = StandardScaler().fit_transform(np.log1p(gasto))

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap, TSNE, trustworthiness

# 1) rollo suizo: una hoja 2D enrollada en 3D
R, t = make_swiss_roll(1500, noise=0.05, random_state=0)
metodos = {"PCA": PCA(2),
           "Isomap": Isomap(n_neighbors=10, n_components=2),
           "t-SNE": TSNE(2, perplexity=30, random_state=0)}
planos = {}
for nombre, met in metodos.items():
    E = planos[nombre] = met.fit_transform(R)
    rho = max(abs(spearmanr(E[:, j], t)[0]) for j in (0, 1))
    print(nombre, "vecinos conservados",
          round(trustworthiness(R, E, n_neighbors=10), 3),
          "orden del rollo", round(rho, 3))

# 2) perfiles de compra de los 440 mayoristas
pca = PCA().fit(X)
print("varianza explicada",
      pca.explained_variance_ratio_.cumsum().round(2))
print(pd.DataFrame(pca.components_[:2].T, index=gasto.columns,
                   columns=["CP1", "CP2"]).round(2))

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(4.4, 1.3))
ax = fig.add_subplot(1, 4, 1, projection="3d")
ax.scatter(R[:, 0], R[:, 1], R[:, 2], c=t, s=1, cmap="viridis")
ax.set_axis_off(); ax.set_title("Rollo suizo")
for i, (nombre, E) in enumerate(planos.items(), start=2):
    ax = fig.add_subplot(1, 4, i)
    ax.scatter(E[:, 0], E[:, 1], c=t, s=1, cmap="viridis")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(nombre)
plt.show()  # fig.savefig("variedades.pdf")

**Prueba.** Cambia `n_neighbors=10` de Isomap por 30. ¿Sigue desplegando el rollo?

## 3. Análisis de la cesta de la compra (3.6)

In [ ]:
import numpy as np, pandas as pd

t = pd.read_csv("cestas.csv")
nombres = pd.read_csv("productos.csv", index_col="producto")
B = pd.crosstab(t.factura, t.producto).clip(upper=1)
n = len(B)
soporte = B.mean()                        # P(A)
juntos = (B.T @ B) / n                    # P(A y B)
juntos.index.name, juntos.columns.name = "si", "entonces"
reglas = juntos.stack().rename("soporte").reset_index()
reglas = reglas[reglas.si != reglas.entonces]
reglas["confianza"] = reglas.soporte / soporte[reglas.si].values
reglas["lift"] = (reglas.confianza
                  / soporte[reglas.entonces].values)
fuertes = reglas[(reglas.soporte >= .01)
                 & (reglas.confianza >= .5)]
print(n, "facturas;", len(fuertes), "reglas fuertes")
for _, r in fuertes.nlargest(6, "lift").iterrows():
    print(nombres.descripcion[r.si], "->",
          nombres.descripcion[r.entonces],
          round(r.soporte, 3), round(r.confianza, 2),
          round(r.lift, 1))
# dos superventas: ¿se compran juntos?
par = reglas[(reglas.si == "22423") & (reglas.entonces == "85123A")]
print(par.round(3).to_string(index=False))

**Ejercicio.** Muestra las cinco reglas con mayor confianza cuyo consecuente sea `85123A`. ¿Tienen todas un lift alto?

> Agrawal, R., Imieliński, T. y Swami, A. (1993). Mining association rules between sets of items in large databases. *Proceedings of the 1993 ACM SIGMOD*, 207–216. https://doi.org/10.1145/170035.170072